# 12 — พล็อตแผนที่อากาศจากข้อมูล ERA5

**คอร์ส Python for Geography** · ภาควิชาภูมิศาสตร์ ชั้นปีที่ 2
**บทเสริม (extension) · ใช้เวลาประมาณ 2–3 ชั่วโมง**

---

### บทนี้อยู่นอกคอร์ส 12 ชั่วโมง

ใช้ได้ 3 แบบ — สอนเป็นคาบที่ 7 ถ้ามีเวลาเพิ่ม, ให้เป็นงานกลุ่มปลายภาค,
หรือให้นักศึกษาที่สนใจต่อยอดไปอ่านเอง

**ต้องผ่านบทที่ 09 (pandas) และ 10 (matplotlib) มาก่อน**

### สิ่งที่จะได้จากบทนี้

- เข้าใจว่า **gridded data** ต่างจากตารางข้อมูลอย่างไร และทำไมต้องใช้เครื่องมือคนละตัว
- เปิดไฟล์ **NetCDF** ด้วย `xarray` เลือกเวลา ระดับความกดอากาศ และตัดโดเมนได้
- วาดแผนที่ด้วย `cartopy` — แรเงา เส้นชั้น wind barb และ streamline
- อ่านแผนที่อากาศเป็น: ความสัมพันธ์ระหว่างสนามความกดอากาศกับลม
- ติดตามการเคลื่อนตัวของระบบด้วยแผนที่หลายช่วงเวลา

---

### ⚠️ เรื่องข้อมูลที่ต้องเข้าใจก่อน

ไฟล์ที่ใช้ในบทนี้ชื่อ `era5_demo_20250914.nc` เป็น **ข้อมูลสังเคราะห์**
สร้างด้วย `utils/make_era5_demo.py` ให้มีโครงสร้างเหมือน ERA5 ทุกอย่าง
— ชื่อตัวแปร มิติ หน่วย attribute ตาม CF convention — **แต่ไม่ใช่ข้อมูลตรวจวัดจริง**

เหตุผลที่ไม่ใช้ ERA5 ตัวจริงในคาบเรียน: ต้องสมัคร Copernicus Climate Data Store
ยอมรับ license แล้วรอคิวดาวน์โหลด ซึ่งทำพร้อมกัน 30 คนในคาบ 2 ชั่วโมงไม่ทัน

**วิธีดึง ERA5 ตัวจริงอยู่ในภาคผนวกท้าย notebook** และเมื่อเปลี่ยนไปใช้ไฟล์จริงแล้ว
โค้ดทุกบรรทัดในบทนี้ใช้ได้เหมือนเดิมไม่ต้องแก้ เพราะชื่อตัวแปรตรงกัน

ห้ามนำผลวิเคราะห์จากข้อมูลสาธิตไปอ้างอิงทางวิชาการเด็ดขาด

## 0. เตรียมเครื่องมือและข้อมูล

Colab มี `xarray` กับ `netCDF4` มาให้แล้ว แต่ไม่มี `cartopy` ต้องติดตั้งเอง
ใช้เวลาประมาณ 1 นาที รันครั้งเดียวต่อ session

In [ ]:
!pip install -q cartopy
print("ติดตั้ง cartopy เรียบร้อย")

In [ ]:
# ===== ดาวน์โหลดข้อมูลและ module ช่วยงาน (รันครั้งเดียวต่อ session) =====
BASE = "https://raw.githubusercontent.com/jamorn12/Teaching_Python_for_Geography/main/"

import os, subprocess, sys, urllib.request

os.makedirs("data", exist_ok=True)
NC = "data/era5_demo_20250914.nc"

if not os.path.exists(NC):
    try:
        urllib.request.urlretrieve(BASE + "data/era5_demo_20250914.nc", NC)
    except Exception as exc:
        print("ดาวน์โหลดไม่ได้:", exc)

if not os.path.exists(NC):
    # สำรอง: สร้างไฟล์ชุดเดียวกันขึ้นมาใหม่
    urllib.request.urlretrieve(BASE + "utils/make_era5_demo.py", "make_era5_demo.py")
    subprocess.run([sys.executable, "make_era5_demo.py", "--outdir", "data"], check=True)

for name in ["era5_utils.py"]:
    if not os.path.exists(name):
        urllib.request.urlretrieve(BASE + "utils/" + name, name)

for name in ["stations.csv"]:
    if not os.path.exists("data/" + name):
        urllib.request.urlretrieve(BASE + "data/" + name, "data/" + name)

print("ไฟล์พร้อม:", sorted(os.listdir("data")))

In [ ]:
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature

import era5_utils as eu

eu.setup_thai_font()
plt.rcParams["figure.dpi"] = 110

## 1. gridded data ต่างจากตารางอย่างไร

ข้อมูลฝนในบทที่ 09 เป็น **ตาราง** หนึ่งแถวคือหนึ่งการตรวจวัดที่หนึ่งจุด หนึ่งเวลา
เหมาะกับข้อมูลสถานีซึ่งกระจายไม่สม่ำเสมอ

ERA5 เป็น **กริด** ค่าถูกวางบนตารางสี่เหลี่ยมที่ครอบคลุมทุกจุดบนโลก
ข้อมูลหนึ่งตัวแปรจึงมี 4 มิติ: `time × level × latitude × longitude`

| | pandas DataFrame | xarray Dataset |
|---|---|---|
| รูปร่าง | 2 มิติ (แถว × คอลัมน์) | กี่มิติก็ได้ |
| เลือกข้อมูลด้วย | ตำแหน่งแถว หรือเงื่อนไข | **ชื่อมิติและค่าพิกัดจริง** |
| เหมาะกับ | ข้อมูลสถานี ข้อมูลสำรวจ | reanalysis, ภาพดาวเทียม, ผลลัพธ์แบบจำลอง |

จุดที่ทำให้ xarray ทำงานสบายคือ **เลือกด้วยค่าจริงได้เลย**
`ds.sel(level=850, latitude=16.5)` ไม่ต้องมานั่งหาว่าอยู่แถวที่เท่าไร

> ถ้าเคยใช้ ArcGIS/QGIS มาก่อน: `xarray.Dataset` คือ **multiband raster ที่มีแกนเวลาและแกนความสูงเพิ่มเข้ามา**

## 2. เปิดไฟล์ NetCDF

`.nc` คือรูปแบบไฟล์มาตรฐานของข้อมูลภูมิอากาศและผลลัพธ์แบบจำลอง
เก็บทั้งตัวเลข พิกัด และคำอธิบาย (metadata) ไว้ในไฟล์เดียว

In [ ]:
ds = xr.open_dataset("data/era5_demo_20250914.nc")
ds

In [ ]:
# ✍️ พิมพ์โค้ดด้านบนด้วยตัวเองที่นี่

กดสามเหลี่ยมหน้าชื่อตัวแปรเพื่อดูรายละเอียดได้ สังเกต 3 ส่วน

- **Dimensions** — ขนาดของแต่ละมิติ
- **Coordinates** — ค่าพิกัดจริงของแต่ละแกน (ลองจิจูด ละติจูด ระดับ เวลา)
- **Data variables** — ตัวแปรอุตุนิยมวิทยาพร้อมหน่วยและคำอธิบาย

In [ ]:
print("มิติ      :", dict(ds.sizes))
print("ตัวแปร    :", list(ds.data_vars))
print("ระดับความกดอากาศ:", ds["level"].values, "hPa")
print("เวลา      :")
for t in ds["time"].values:
    print("   ", np.datetime_as_string(t, unit="m"))
print("โดเมน     : lon", float(ds.longitude.min()), "ถึง", float(ds.longitude.max()),
      "| lat", float(ds.latitude.min()), "ถึง", float(ds.latitude.max()))

In [ ]:
# ✍️ พิมพ์โค้ดด้านบนด้วยตัวเองที่นี่

### ตัวแปรที่มีในไฟล์

ชื่อย่อเหล่านี้เป็นชื่อมาตรฐานของ ECMWF ควรจำไว้ เพราะเจอทุกครั้งที่ใช้ ERA5

| ชื่อ | ความหมาย | หน่วยในไฟล์ | หน่วยที่คนอ่าน |
|---|---|---|---|
| `t2m` | อุณหภูมิที่ความสูง 2 เมตร | K | °C |
| `msl` | ความกดอากาศที่ระดับน้ำทะเล | Pa | hPa |
| `tp` | ปริมาณฝนสะสม | m | mm |
| `u` | ลมในแนวตะวันออก–ตะวันตก | m s⁻¹ | m s⁻¹ |
| `v` | ลมในแนวเหนือ–ใต้ | m s⁻¹ | m s⁻¹ |
| `z` | geopotential | m² s⁻² | gpm (หาร 9.80665) |
| `r` | ความชื้นสัมพัทธ์ | % | % |

⚠️ **`u` เป็นบวก = ลมพัดไปทางตะวันออก** (westerly) ไม่ใช่พัดมาจากตะวันออก
เป็นจุดที่สับสนกันบ่อยที่สุดเวลาอ่านข้อมูลลม

In [ ]:
t2m = ds["t2m"]

print(type(t2m))
print("มิติ :", t2m.dims)
print("รูปร่าง:", t2m.shape)
print("หน่วย :", t2m.attrs["units"])
print("ชื่อเต็ม:", t2m.attrs["long_name"])
print()
print("attribute ของทั้งไฟล์:")
for k, v in ds.attrs.items():
    print(f"  {k}: {v}")

In [ ]:
# ✍️ พิมพ์โค้ดด้านบนด้วยตัวเองที่นี่

## 3. การเลือกข้อมูล: `sel` และ `isel`

- `.sel()` เลือกด้วย **ค่าจริง** เช่น `level=850`, `time="2025-09-14T12:00"`
- `.isel()` เลือกด้วย **ลำดับ** เช่น `time=0` คือเวลาแรก
- `method="nearest"` ให้เลือกค่าที่ใกล้ที่สุด เมื่อค่าที่ขอไม่ตรงกริดพอดี

ส่วนใหญ่ใช้ `.sel()` เพราะอ่านแล้วรู้เลยว่าเลือกอะไร

In [ ]:
# เลือกเวลาเดียว
t_one = ds["t2m"].sel(time="2025-09-14T12:00")
print("หลังเลือกเวลา:", t_one.dims, t_one.shape)

# เลือกระดับความกดอากาศ
u850 = ds["u"].sel(level=850)
print("ลมที่ 850 hPa  :", u850.dims, u850.shape)

# เลือกทั้งเวลาและระดับ -> เหลือ 2 มิติ พร้อมพล็อตเป็นแผนที่
u850_t12 = ds["u"].sel(time="2025-09-14T12:00", level=850)
print("พร้อมพล็อต     :", u850_t12.dims, u850_t12.shape)

# ค่าที่จุดเดียว (พิกัดสถานีชลบุรี) ใช้ nearest เพราะพิกัดไม่ตรงกริดพอดี
point = ds["t2m"].sel(latitude=13.367, longitude=100.983, method="nearest")
print("\nอุณหภูมิที่จุดใกล้ชลบุรี ทุกช่วงเวลา (°C):")
print((point - 273.15).round(2).values)

In [ ]:
# ✍️ พิมพ์โค้ดด้านบนด้วยตัวเองที่นี่

### ตัดโดเมน (subset) ด้วย `slice`

⚠️ จุดที่พลาดกันบ่อย: ไฟล์ ERA5 ส่วนใหญ่เรียง `latitude` **จากมากไปน้อย**
ถ้าสั่ง `slice(5, 21)` กับไฟล์แบบนั้นจะได้ข้อมูลเปล่า ต้องสลับเป็น `slice(21, 5)`

ทางที่ปลอดภัยกว่าคือเรียงใหม่ให้เป็นน้อยไปมากก่อน แล้วค่อย slice ตามปกติ
— ฟังก์ชัน `eu.open_era5()` ทำขั้นนี้ให้อัตโนมัติ

In [ ]:
print("latitude 5 ค่าแรกในไฟล์ดิบ:", ds["latitude"].values[:5])

ds = eu.open_era5("data/era5_demo_20250914.nc")   # เรียง latitude ให้เรียบร้อย
print("หลัง open_era5()        :", ds["latitude"].values[:5])

thai = ds.sel(longitude=slice(96, 106.5), latitude=slice(5, 21))
print("\nโดเมนประเทศไทย:", dict(thai.sizes))

# หรือใช้ชื่อโดเมนสำเร็จรูป
thai2 = eu.subset(ds, domain="thailand")
print("เท่ากันไหม:", thai.sizes == thai2.sizes)
print("โดเมนที่มีให้เลือก:", list(eu.DOMAINS))

In [ ]:
# ✍️ พิมพ์โค้ดด้านบนด้วยตัวเองที่นี่

## 4. แปลงหน่วยก่อนพล็อต

ไม่มีใครอ่านแผนที่ที่บอกอุณหภูมิเป็นเคลวินหรือความกดอากาศเป็นปาสกาล
แปลงให้เป็นหน่วยที่คนใช้งานจริงเสมอ **และเขียนหน่วยกำกับไว้บนแผนที่ด้วย**

In [ ]:
snap = eu.subset(ds, time="2025-09-14T12:00")

t_c = eu.to_celsius(snap["t2m"])
p_hpa = eu.to_hpa(snap["msl"])
rain_mm = eu.to_mm(snap["tp"])
z500_gpm = eu.to_gpm(snap["z"].sel(level=500))

print(f"อุณหภูมิ      {float(t_c.min()):6.1f} ถึง {float(t_c.max()):6.1f} °C")
print(f"ความกดอากาศ  {float(p_hpa.min()):6.1f} ถึง {float(p_hpa.max()):6.1f} hPa")
print(f"ฝน           {float(rain_mm.min()):6.1f} ถึง {float(rain_mm.max()):6.1f} mm")
print(f"ความสูง 500  {float(z500_gpm.min()):6.0f} ถึง {float(z500_gpm.max()):6.0f} gpm")

# ค่าเฉลี่ยเชิงพื้นที่ต้องถ่วงน้ำหนักด้วย cos(latitude)
print(f"\nอุณหภูมิเฉลี่ยทั้งโดเมน (ถ่วงน้ำหนัก) = {eu.area_mean(t_c):.2f} °C")
print(f"ถ้าเฉลี่ยตรง ๆ ไม่ถ่วงน้ำหนัก        = {float(t_c.mean()):.2f} °C")

In [ ]:
# ✍️ พิมพ์โค้ดด้านบนด้วยตัวเองที่นี่

> **ทำไมต้องถ่วงน้ำหนักด้วย cos(latitude)**
> กริดละติจูด–ลองจิจูดมีเซลล์ขนาดไม่เท่ากัน เซลล์ใกล้ขั้วโลกแคบกว่าเซลล์ใกล้ศูนย์สูตร
> ถ้าเฉลี่ยตรง ๆ จะให้น้ำหนักพื้นที่ขั้วโลกเกินจริง
> ในโดเมนเขตร้อนแคบ ๆ ต่างกันไม่มาก แต่พอทำค่าเฉลี่ยทั้งโลกจะต่างกันอย่างมีนัยสำคัญ

## 5. แผนที่ใบแรก

ขั้นตอนของแผนที่หนึ่งใบมี 4 ส่วนเสมอ

1. สร้างแกนที่มี **projection** — `ccrs.PlateCarree()` คือพิกัดภูมิศาสตร์ธรรมดา
2. กำหนดขอบเขต + ใส่เส้นชายฝั่ง เส้นพรมแดน gridline
3. วาดข้อมูลทับลงไป โดย **ต้องระบุ `transform=ccrs.PlateCarree()`** เสมอ
4. ใส่ชื่อแผนที่ colorbar แล้วบันทึก

> `transform=` บอก cartopy ว่า **ข้อมูลอยู่ในระบบพิกัดอะไร**
> ส่วน projection ของแกนคือ **ระบบที่จะวาดออกมา** ถ้าลืม `transform` cartopy
> จะเดาว่าข้อมูลอยู่ในระบบเดียวกับแกน ซึ่งพอเปลี่ยน projection เมื่อไรแผนที่จะเพี้ยนทันที

In [ ]:
fig = plt.figure(figsize=(9, 7))
ax = fig.add_subplot(1, 1, 1, projection=ccrs.PlateCarree())

ax.set_extent([90, 115, 0, 25], crs=ccrs.PlateCarree())

# cartopy ไม่ได้ติดตั้งแผนที่ฐานมาด้วย แต่ไปโหลดจาก Natural Earth ตอนวาดรูป
# ถ้าเครือข่ายบล็อก จะพังตอน plt.show() ซึ่งดักยาก จึงบังคับโหลดตรงนี้ก่อน
try:
    for feat in (cfeature.COASTLINE, cfeature.BORDERS, cfeature.OCEAN, cfeature.LAND):
        next(iter(feat.geometries()), None)
    ax.add_feature(cfeature.OCEAN, facecolor="#eaf3fb")
    ax.add_feature(cfeature.LAND, facecolor="#f7f5f0")
    ax.add_feature(cfeature.COASTLINE, linewidth=0.8)
    ax.add_feature(cfeature.BORDERS, linewidth=0.5, linestyle=":")
except Exception as exc:
    print("โหลดแผนที่ฐานไม่ได้ ->", exc)
    print("แผนที่จะไม่มีเส้นชายฝั่ง แต่ข้อมูลที่พล็อตยังถูกต้องทุกอย่าง")

gl = ax.gridlines(draw_labels=True, linewidth=0.4, color="gray",
                  alpha=0.5, linestyle="--")
gl.top_labels = False
gl.right_labels = False

cf = ax.contourf(t_c["longitude"], t_c["latitude"], t_c.values,
                 levels=np.arange(18, 34.1, 1), cmap="RdYlBu_r",
                 extend="both", transform=ccrs.PlateCarree())

cb = plt.colorbar(cf, ax=ax, shrink=0.8, pad=0.04)
cb.set_label("อุณหภูมิที่ 2 เมตร (°C)")
ax.set_title("ERA5 (สาธิต) — อุณหภูมิ 2 เมตร\n2025-09-14 12 UTC", loc="left")
plt.show()

In [ ]:
# ✍️ พิมพ์โค้ดด้านบนด้วยตัวเองที่นี่

ถ้าขึ้นข้อความว่าโหลดเส้นชายฝั่งไม่ได้ แปลว่าเครือข่ายบล็อกการดาวน์โหลดข้อมูล Natural Earth
แผนที่จะไม่มีเส้นชายฝั่ง แต่ **ข้อมูลที่พล็อตยังถูกต้องทุกอย่าง** ใช้เรียนต่อได้

## 6. แผนที่ความกดอากาศ — isobar

เส้นชั้นความกดอากาศ (isobar) คือหัวใจของแผนที่อากาศผิวพื้น
ธรรมเนียมสากลคือวาดทุก **4 hPa** และเขียนตัวเลขกำกับเส้น

In [ ]:
ax = eu.make_map_axes(domain="indochina", figsize=(9, 7))

cs = ax.contour(p_hpa["longitude"], p_hpa["latitude"], p_hpa.values,
                levels=np.arange(980, 1021, 4), colors="black",
                linewidths=1.0, transform=ccrs.PlateCarree())
ax.clabel(cs, inline=True, fontsize=9, fmt="%d")

# ทำเครื่องหมายศูนย์กลางความกดอากาศต่ำ
low = eu.find_low_center(p_hpa)
ax.plot(low["lon"], low["lat"], marker="x", color="red", markersize=14,
        markeredgewidth=3, transform=ccrs.PlateCarree())
ax.text(low["lon"] + 0.4, low["lat"] + 0.4, f"L {low['value']:.0f}",
        color="red", fontweight="bold", transform=ccrs.PlateCarree())

ax.set_title("ความกดอากาศที่ระดับน้ำทะเล (hPa)\n2025-09-14 12 UTC", loc="left")
plt.show()

print(f"ศูนย์กลาง: {low['lat']:.2f}N {low['lon']:.2f}E  {low['value']:.1f} hPa")

In [ ]:
# ✍️ พิมพ์โค้ดด้านบนด้วยตัวเองที่นี่

## 7. ลม — wind barb

`barbs()` วาดสัญลักษณ์ลมแบบที่ใช้ในแผนที่อากาศจริง

- ก้านชี้ไป **ทิศที่ลมพัดมาจาก**
- ขีดสั้น = 5 นอต · ขีดยาว = 10 นอต · ธงสามเหลี่ยม = 50 นอต

**ต้องแปลงจาก m/s เป็นนอตก่อนเสมอ** (× 1.94384) เพราะ barb ถูกนิยามด้วยนอต
ถ้าใส่ m/s ตรง ๆ จะอ่านค่าผิดไปเกือบเท่าตัว

และต้องเว้นช่วงกริด (`step`) ไม่งั้นลูกศรจะทับกันจนอ่านไม่ออก

In [ ]:
lev = 850
d = eu.subset(ds, time="2025-09-14T12:00", level=lev)
ws = eu.wind_speed(d["u"], d["v"])

ax = eu.make_map_axes(domain="indochina", figsize=(9.5, 7.5))

cf = eu.shade(ax, ws, cmap="YlGnBu", levels=np.arange(0, 31, 2.5))
eu.barbs(ax, d["u"], d["v"], step=8, color="black")

eu.finish(ax, title=f"ลมที่ระดับ {lev} hPa — ความเร็ว (แรเงา) และทิศทาง (barb)\n2025-09-14 12 UTC",
          mappable=cf, cbar_label="ความเร็วลม (m s$^{-1}$)")
plt.show()

print(f"ลมแรงสุดในโดเมน {float(ws.max()):.1f} m/s = {float(ws.max())*1.94384:.0f} นอต")

In [ ]:
# ✍️ พิมพ์โค้ดด้านบนด้วยตัวเองที่นี่

### อ่านแผนที่: ลมสัมพันธ์กับความกดอากาศอย่างไร

ลองดูสองแผนที่ข้างบนคู่กัน จะเห็นว่า**ลมพัดวนทวนเข็มนาฬิการอบศูนย์กลางความกดอากาศต่ำ**

นี่คือผลของ **Coriolis force** ในซีกโลกเหนือ อากาศไม่ได้ไหลตรงจากความกดอากาศสูง
ไปต่ำเหมือนน้ำไหลลงที่ต่ำ แต่ถูกเบี่ยงไปทางขวาจนกลายเป็นการไหลเกือบขนานกับ isobar
(**geostrophic balance**)

สองข้อสังเกตที่ใช้อ่านแผนที่อากาศได้ทันที
- isobar อยู่ชิดกัน = ความกดอากาศเปลี่ยนเร็ว (pressure gradient แรง) = **ลมแรง**
- ซีกโลกใต้หมุนกลับทาง เพราะ Coriolis เบี่ยงไปทางซ้าย

## 8. แผนที่ระดับบน — 500 hPa

แผนที่ 500 hPa คือแผนที่ที่นักอุตุนิยมวิทยาดูบ่อยที่สุดรองจากผิวพื้น
เพราะอยู่ประมาณกึ่งกลางมวลบรรยากาศ บอกทิศทางการเคลื่อนตัวของระบบอากาศได้ดี

สิ่งที่พล็อตคือ **geopotential height** (หน่วย gpm) ไม่ใช่ความกดอากาศ
— ที่ระดับความกดอากาศคงที่ เราดูว่า "ระดับนั้นอยู่สูงเท่าไร" แทน
ค่าต่ำ = อากาศเย็น/ร่องความกดอากาศต่ำ (trough) · ค่าสูง = สันความกดอากาศสูง (ridge)

In [ ]:
d500 = eu.subset(ds, time="2025-09-14T12:00", level=500)
z500 = eu.to_gpm(d500["z"])
ws500 = eu.wind_speed(d500["u"], d500["v"])

ax = eu.make_map_axes(domain="indochina", figsize=(9.5, 7.5))

cf = eu.shade(ax, ws500, cmap="BuPu", levels=np.arange(0, 25, 2))
eu.contour(ax, z500, levels=np.arange(5700, 5951, 15), color="darkred",
           linewidth=1.1, label_fmt="%d")
eu.barbs(ax, d500["u"], d500["v"], step=10, color="black")

eu.finish(ax, title="500 hPa — geopotential height (เส้น, gpm) และลม\n2025-09-14 12 UTC",
          mappable=cf, cbar_label="ความเร็วลม (m s$^{-1}$)")
plt.show()

In [ ]:
# ✍️ พิมพ์โค้ดด้านบนด้วยตัวเองที่นี่

## 9. ความชื้นและฝน

ที่ 850 hPa เป็นระดับที่ใช้ดูการนำพาความชื้น (moisture transport) เพราะอยู่เหนือ
ชั้นที่ได้รับผลจากภูมิประเทศโดยตรง แต่ยังอยู่ในชั้นที่มีความชื้นมาก

คราวนี้ใช้ **streamline** แทน barb — เหมาะกับเขตร้อนที่สนใจ *รูปแบบการไหล*
มากกว่าความเร็วลมแต่ละจุด

In [ ]:
d850 = eu.subset(ds, time="2025-09-14T12:00", level=850)

ax = eu.make_map_axes(domain="indochina", figsize=(9.5, 7.5))

cf = eu.shade(ax, d850["r"], cmap="BrBG", levels=np.arange(20, 101, 5))
eu.streamlines(ax, d850["u"], d850["v"], density=1.6)

eu.finish(ax, title="850 hPa — ความชื้นสัมพัทธ์ (แรเงา) และเส้นกระแสลม\n2025-09-14 12 UTC",
          mappable=cf, cbar_label="ความชื้นสัมพัทธ์ (%)")
plt.show()

In [ ]:
# ✍️ พิมพ์โค้ดด้านบนด้วยตัวเองที่นี่

In [ ]:
# ฝนสะสม — ใช้ colormap ที่มีค่าต่ำโปร่งใสเพื่อให้เห็นแผนที่ข้างล่าง
ax = eu.make_map_axes(domain="indochina", figsize=(9.5, 7.5))

levels = [0.5, 1, 2, 5, 10, 15, 20, 25, 30]
cf = eu.shade(ax, rain_mm, cmap="GnBu", levels=levels, extend="max")
eu.contour(ax, p_hpa, levels=np.arange(980, 1021, 4), color="gray",
           linewidth=0.7, label_fmt="%d", inline_fontsize=7)

eu.finish(ax, title="ฝนสะสม 6 ชั่วโมง (แรเงา) และความกดอากาศผิวพื้น\n2025-09-14 12 UTC",
          mappable=cf, cbar_label="ปริมาณฝน (mm)")
plt.show()

In [ ]:
# ✍️ พิมพ์โค้ดด้านบนด้วยตัวเองที่นี่

## 10. ติดตามการเคลื่อนตัว — แผนที่หลายช่วงเวลา

แผนที่ใบเดียวบอกได้แค่ว่า "ตอนนี้เป็นอย่างไร"
การวางหลายเวลาเรียงกันทำให้เห็น **ทิศทางและความเร็วการเคลื่อนตัว** ซึ่งคือสิ่งที่ใช้พยากรณ์

ข้อสำคัญ: ทุกช่องต้องใช้ **ระดับสีและเส้นชั้นชุดเดียวกัน** ไม่งั้นเปรียบเทียบไม่ได้

In [ ]:
times = ds["time"].values
fig, axes = plt.subplots(2, 2, figsize=(13, 10),
                         subplot_kw={"projection": ccrs.PlateCarree()})

levels = np.arange(980, 1021, 2)
track = []

for ax, t in zip(axes.flat, times):
    snap_t = eu.subset(ds, time=t)
    p_t = eu.to_hpa(snap_t["msl"])
    rain_t = eu.to_mm(snap_t["tp"])

    eu.make_map_axes(ax=ax, domain="indochina")
    cf = eu.shade(ax, rain_t, cmap="GnBu", levels=[0.5, 2, 5, 10, 20, 30], extend="max")
    eu.contour(ax, p_t, levels=levels, color="black", linewidth=0.7,
               label_fmt="%d", inline_fontsize=7)

    low = eu.find_low_center(p_t)
    track.append((np.datetime_as_string(t, unit="h"), low))
    ax.plot(low["lon"], low["lat"], "rx", markersize=12, markeredgewidth=2.5,
            transform=ccrs.PlateCarree())

    ax.set_title(np.datetime_as_string(t, unit="m") + " UTC", fontsize=11, loc="left")

fig.colorbar(cf, ax=axes, orientation="horizontal", shrink=0.5,
             pad=0.05, label="ฝนสะสม 6 ชั่วโมง (mm)")
fig.suptitle("การเคลื่อนตัวของหย่อมความกดอากาศต่ำ 14 ก.ย. 2025", fontsize=14)
plt.show()

print("เส้นทางศูนย์กลาง")
for stamp, low in track:
    print(f"  {stamp}  {low['lat']:5.2f}N {low['lon']:6.2f}E  {low['value']:7.1f} hPa")

In [ ]:
# ✍️ พิมพ์โค้ดด้านบนด้วยตัวเองที่นี่

## 11. ซูมเข้าพื้นที่ศึกษา และดึงค่าที่จุดสถานี

งานภูมิศาสตร์มักต้องตอบว่า "ที่สถานีของเรา ค่าเป็นเท่าไร"
`sel(..., method="nearest")` ดึงค่าจากกริดที่ใกล้พิกัดนั้นที่สุดให้ได้ทันที

In [ ]:
stations = pd.read_csv("data/stations.csv")

ax = eu.make_map_axes(domain="east", figsize=(8, 7))
d_east = eu.subset(ds, domain="east", time="2025-09-14T12:00", level=850)

cf = eu.shade(ax, eu.to_celsius(d_east["t2m"]), cmap="RdYlBu_r",
              levels=np.arange(24, 32.1, 0.5))
eu.barbs(ax, d_east["u"], d_east["v"], step=3, color="black")
eu.add_station_markers(ax, stations)

eu.finish(ax, title="ซูมพื้นที่ชายฝั่งตะวันออก — อุณหภูมิ 2 เมตร และลม 850 hPa",
          mappable=cf, cbar_label="°C")
plt.show()

In [ ]:
# ✍️ พิมพ์โค้ดด้านบนด้วยตัวเองที่นี่

In [ ]:
# ดึงค่าที่พิกัดสถานีทุกแห่ง ทุกช่วงเวลา แล้วจัดเป็นตาราง
rows = []
for _, s in stations.iterrows():
    point = ds.sel(latitude=s["lat"], longitude=s["lon"], method="nearest")
    for t in ds["time"].values:
        p = point.sel(time=t)
        rows.append({
            "station_id": s["station_id"],
            "station_name": s["station_name"],
            "time": pd.Timestamp(t),
            "t2m_c": round(float(p["t2m"]) - 273.15, 2),
            "msl_hpa": round(float(p["msl"]) / 100, 1),
            "rain_mm": round(float(p["tp"]) * 1000, 2),
            "ws850": round(float(np.sqrt(p["u"].sel(level=850)**2
                                         + p["v"].sel(level=850)**2)), 1),
        })

point_df = pd.DataFrame(rows)
display(point_df.head(8))

print("\nสรุปรายสถานี")
print(point_df.groupby("station_name").agg(
    t2m_mean=("t2m_c", "mean"),
    msl_min=("msl_hpa", "min"),
    rain_total=("rain_mm", "sum"),
    ws850_max=("ws850", "max"),
).round(2).to_string())

In [ ]:
# ✍️ พิมพ์โค้ดด้านบนด้วยตัวเองที่นี่

จุดนี้คือสะพานเชื่อมระหว่างบทที่ 09 กับบทนี้ — แปลงข้อมูล **กริด** ให้กลับเป็น **ตาราง**
แล้ววิเคราะห์ต่อด้วย pandas ที่เรียนมาแล้วได้เลย

## 12. บันทึกรูปสำหรับรายงาน

In [ ]:
ax = eu.make_map_axes(domain="thailand", figsize=(7.5, 9))
d_th = eu.subset(ds, domain="thailand", time="2025-09-14T12:00", level=850)

cf = eu.shade(ax, eu.wind_speed(d_th["u"], d_th["v"]), cmap="YlGnBu",
              levels=np.arange(0, 26, 2))
eu.contour(ax, eu.to_hpa(d_th["msl"]), levels=np.arange(980, 1021, 2),
           color="black", linewidth=0.8)
eu.barbs(ax, d_th["u"], d_th["v"], step=4)

eu.finish(ax, title="ประเทศไทย — ลม 850 hPa และความกดอากาศผิวพื้น\n2025-09-14 12 UTC",
          mappable=cf, cbar_label="ความเร็วลม (m s$^{-1}$)",
          savepath="era5_thailand_850hPa.png", dpi=300)
plt.show()
!ls -lh era5_thailand_850hPa.png

## 13. แบบฝึกหัดท้ายบท

1. พล็อตแผนที่ **อุณหภูมิ 2 เมตร** เวลา `2025-09-14T00:00` เทียบกับ `2025-09-14T06:00`
   วางคู่กันด้วย `subplots(1, 2)` ใช้ระดับสีเดียวกัน แล้วอธิบายว่าต่างกันเพราะอะไร
2. หา **ระดับความกดอากาศที่ลมแรงที่สุด** โดยคำนวณความเร็วลมเฉลี่ยทั้งโดเมนของทุก level
   แล้วพล็อตกราฟเส้น (แกน y เป็นระดับความกดอากาศ เรียงจากมากไปน้อยให้เหมือนหน้าตัดบรรยากาศจริง)
3. พล็อต **หน้าตัดแนวดิ่ง** (vertical cross-section) ของความชื้นสัมพัทธ์ตามแนวละติจูด 16.5°N
   (ใบ้: `ds["r"].sel(latitude=16.5, method="nearest")` จะเหลือมิติ `level × longitude`)
4. คำนวณ **ฝนสะสมรวมทั้งวัน** (รวม 4 ช่วงเวลา) แล้วพล็อตเป็นแผนที่เดียว
   พร้อมทำเครื่องหมายจุดที่ฝนตกหนักที่สุดและพิมพ์พิกัดออกมา

In [ ]:
# ข้อ 1

In [ ]:
# ข้อ 2

In [ ]:
# ข้อ 3

In [ ]:
# ข้อ 4

---

### เฉลยแบบฝึกหัด

ผู้สอนจะแจกเฉลยหลังหมดเวลาทำแบบฝึกหัดในคาบ

> ลองด้วยตัวเองให้สุดก่อน การติดแล้วแก้ได้เองคือช่วงที่สมองเรียนรู้มากที่สุด
> ถ้าติดเกิน 10 นาทีให้ยกมือถามได้เลย ไม่ต้องรอ


---

## ภาคผนวก — ดึงข้อมูล ERA5 ตัวจริง

เมื่อพร้อมใช้ข้อมูลจริง ขั้นตอนมีดังนี้ ทำครั้งเดียวแล้วใช้ได้ตลอด

### 1. สมัครและรับ API key

1. สมัครบัญชีที่ [cds.climate.copernicus.eu](https://cds.climate.copernicus.eu)
2. เข้าหน้า profile ของตัวเอง copy **UID** และ **API key**
3. เข้าหน้า dataset ที่ต้องการ แล้วกดยอมรับ license ที่ท้ายหน้า
   — **ข้อนี้ลืมกันบ่อยที่สุด** ถ้าไม่กด จะดาวน์โหลดไม่ได้และ error ไม่บอกสาเหตุชัด

### 2. ตั้งค่าใน Colab

```python
!pip install -q cdsapi

with open("/root/.cdsapirc", "w") as f:
    f.write("url: https://cds.climate.copernicus.eu/api\n")
    f.write("key: ใส่ API key ของคุณที่นี่\n")
```

### 3. สั่งดาวน์โหลด

```python
import cdsapi

c = cdsapi.Client()

# ---- ตัวแปรที่ระดับความกดอากาศ ----
c.retrieve(
    "reanalysis-era5-pressure-levels",
    {
        "product_type": "reanalysis",
        "format": "netcdf",
        "variable": [
            "geopotential", "relative_humidity", "temperature",
            "u_component_of_wind", "v_component_of_wind",
        ],
        "pressure_level": ["1000", "925", "850", "700", "500", "200"],
        "year": "2025",
        "month": "09",
        "day": ["14"],
        "time": ["00:00", "06:00", "12:00", "18:00"],
        "area": [25, 90, 0, 115],      # เหนือ ตะวันตก ใต้ ตะวันออก
    },
    "era5_pressure_levels.nc",
)

# ---- ตัวแปรที่ผิวพื้น ----
c.retrieve(
    "reanalysis-era5-single-levels",
    {
        "product_type": "reanalysis",
        "format": "netcdf",
        "variable": [
            "2m_temperature", "mean_sea_level_pressure", "total_precipitation",
        ],
        "year": "2025",
        "month": "09",
        "day": ["14"],
        "time": ["00:00", "06:00", "12:00", "18:00"],
        "area": [25, 90, 0, 115],
    },
    "era5_single_levels.nc",
)
```

### 4. ใช้กับโค้ดในบทนี้

```python
ds_pl = eu.open_era5("era5_pressure_levels.nc")
ds_sl = eu.open_era5("era5_single_levels.nc")
ds_real = xr.merge([ds_pl, ds_sl])
```

จากนั้นแทนที่ `ds` ด้วย `ds_real` แล้วรันเซลล์เดิมได้ทั้งหมด

### ข้อควรรู้เมื่อใช้ข้อมูลจริง

- **เวลาเป็น UTC เสมอ** เวลาไทย = UTC + 7 ชั่วโมง ต้องบวกเองทุกครั้งที่ตีความ
- `area` เรียงลำดับ **เหนือ ตะวันตก ใต้ ตะวันออก** ไม่ใช่ lon ก่อน lat
- `tp` ของ ERA5 เป็น **ฝนสะสมรายชั่วโมง** หน่วยเมตร ต้องรวมเองถ้าต้องการรายวัน
- คิวดาวน์โหลดอาจรอ 5 นาทีถึงหลายชั่วโมง แล้วแต่ขนาดคำขอและความหนาแน่นของผู้ใช้
- ขอเฉพาะโดเมนและตัวแปรที่ใช้จริง ขอทั้งโลกทุกตัวแปรจะได้ไฟล์หลายสิบ GB

### อ้างอิง

Hersbach, H. et al. (2020). The ERA5 global reanalysis.
*Quarterly Journal of the Royal Meteorological Society*, 146(730), 1999–2049.

เงื่อนไขการใช้งานกำหนดให้ **ต้องอ้างอิงแหล่งข้อมูล** เมื่อนำ ERA5 ไปใช้ในงานวิชาการ

### ไปต่อทางไหนได้อีก

- **MetPy** — คำนวณตัวแปรอุตุนิยมวิทยา (CAPE, vorticity, frontogenesis) จากข้อมูลกริด
- **GeoPandas** — ซ้อนขอบเขตจังหวัดลงบนแผนที่ แล้วหาค่าเฉลี่ยรายจังหวัด
- **xarray + dask** — จัดการข้อมูลหลายสิบปีที่ใหญ่เกินหน่วยความจำ